## 0. Cài đặt chung + mount Drive (để không mất kết quả khi Colab ngắt session)

In [ ]:
!pip install -q jiwer soundfile librosa pandas speechbrain torchaudio underthesea

from google.colab import drive
drive.mount('/content/drive')

import os
OUT_ROOT = "/content/drive/MyDrive/tts_eval"
AUDIO_DIR = f"{OUT_ROOT}/audio"
RESULT_DIR = f"{OUT_ROOT}/results"
os.makedirs(AUDIO_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)
print("Output sẽ lưu tại:", OUT_ROOT)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.1/782.1 kB 51.9 MB/s eta 0:00:00
Mounted at /content/drive
Output sẽ lưu tại: /content/drive/MyDrive/tts_eval


## 1. Bộ prompt tiếng Việt

24 câu, chia nhóm: cảm thán, cảm xúc/đối thoại, số-tên riêng-viết tắt, câu dài, đúng ngữ pháp, vô nghĩa ngữ nghĩa.

In [ ]:
PROMPTS = {
    "exclaim_01": "Wow, đội tuyển vừa ghi bàn thắng đẹp quá!",
    "exclaim_02": "Trời ơi, xem pháo hoa tối qua đẹp không thể tin nổi!",
    "emotion_01": "Anh xin lỗi... anh thực sự xin lỗi, nhưng chuyện đã như vậy rồi, anh nói trong nghẹn ngào.",
    "emotion_02": "Em buồn lắm, không biết phải làm sao để vượt qua chuyện này nữa.",
    "tech_01": "Vui lòng đăng nhập với tên người dùng Admin_2026 và mật khẩu Vi3tNam#88.",
    "tech_02": "Tọa độ tập kết là 10 độ 46 phút Bắc, 106 độ 42 phút Đông, có mặt lúc 0300 giờ.",
    "number_01": "Hóa đơn hôm nay là một triệu hai trăm năm mươi nghìn đồng, giảm 15 phần trăm so với tuần trước.",
    "abbrev_01": "TP.HCM và Hà Nội là hai thành phố lớn nhất Việt Nam, theo báo cáo của Tổng cục Thống kê.",
    "long_01": "Sau khi hoàn thành bài kiểm tra, sinh viên cần nộp bài qua hệ thống trực tuyến trước 23 giờ 59 phút, đồng thời lưu lại một bản sao để đối chiếu khi cần thiết.",
    "long_02": "Dự án tốt nghiệp của tôi tập trung vào việc đánh giá và triển khai mô hình chuyển văn bản thành giọng nói cho tiếng Việt, nhằm phục vụ mục đích nghiên cứu và ứng dụng thực tế.",
    "question_01": "Bạn đã ăn cơm chưa? Nếu chưa thì mình cùng đi ăn nhé.",
    "question_02": "Tại sao hôm nay trời lại mưa to như vậy nhỉ?",
    "everyday_01": "Hôm nay thời tiết khá đẹp, rất thích hợp để đi dạo công viên.",
    "everyday_02": "Cà phê sữa đá là thức uống yêu thích của rất nhiều người Việt Nam.",
    "sus_01": "Con mèo đang đọc quyển sách trong nhà bếp một cách chậm rãi.",
    "sus_02": "Chiếc xe đạp màu tím bay qua ngọn núi vào buổi trưa hôm qua.",
    "sus_03": "Bàn ghế trong lớp học bỗng nhiên hát một bài ca cổ xưa.",
    "dialogue_01": "\"Cậu có chắc là đường này đúng không?\", cô ấy hỏi với giọng lo lắng.",
    "dialogue_02": "\"Được rồi, để tôi thử lại lần nữa\", anh ta nói rồi thở dài.",
    "narration_01": "Ngày xửa ngày xưa, ở một ngôi làng nhỏ ven sông, có một cậu bé tên là Tí.",
    "news_01": "Theo dự báo, nhiệt độ tại Hà Nội ngày mai sẽ giảm còn mười tám độ C do ảnh hưởng của không khí lạnh.",
    "polite_01": "Xin vui lòng chờ trong giây lát, nhân viên sẽ hỗ trợ quý khách ngay sau đây.",
    "list_01": "Danh sách cần mua gồm có: gạo, trứng, rau xanh, và một ít thịt heo.",
    "mixed_lang_01": "Chúng ta sẽ deploy model này lên server bằng FastAPI trong tuần tới.",
}

REF_AUDIO_PATH = f"{OUT_ROOT}/ref_audio.wav"  # tự upload 1 file giọng thật (mono, 16/24kHz, 3-10s) vào đây trước khi chạy phần SIM-o
print(f"Tổng số câu prompt: {len(PROMPTS)}")

Tổng số câu prompt: 24


### Lấy ref_audio nhanh từ dataset công khai (VIVOS)

Thay vì tự thu âm, lấy 1 câu mẫu có sẵn từ bộ VIVOS (giọng thật, sạch, có transcript đi kèm) qua `datasets` của HuggingFace.

In [ ]:
!pip install -q datasets

from datasets import load_dataset
import soundfile as sf

vivos = load_dataset("AILAB-VNUHCM/vivos", split="test", revision="refs/convert/parquet")

sample = None
for row in vivos:
    dur = len(row["audio"]["array"]) / row["audio"]["sampling_rate"]
    if 3 <= dur <= 10:
        sample = row
        break

sf.write(REF_AUDIO_PATH, sample["audio"]["array"], sample["audio"]["sampling_rate"])
print("Đã lưu ref_audio tại:", REF_AUDIO_PATH)
print("Transcript của câu mẫu:", sample["sentence"])


default/train/0000.parquet: reconstructing file:   0%|          |  0.00B /  502MB            

default/train/0000.parquet: downloading bytes:           |  0.00B            

default/train/0001.parquet: reconstructing file:   0%|          |  0.00B /  506MB            

default/train/0001.parquet: downloading bytes:           |  0.00B            

default/train/0002.parquet: reconstructing file:   0%|          |  0.00B /  497MB            

...

default/test/0000.parquet: reconstructing file:   0%|          |  0.00B / 85.0MB            

default/test/0000.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Đã lưu ref_audio tại: /content/drive/MyDrive/tts_eval/ref_audio.wav
Transcript của câu mẫu: THẾ NHƯNG KHI GIÁ PHÔI THÉP THẾ GIỚI CAO DẦN


## 2. Hàm chung

Mỗi model implement 1 hàm `generate_<model>(text, out_path)`. Audio lưu tại `AUDIO_DIR/<model_id>/<prompt_id>.wav`.

In [ ]:
import soundfile as sf
import numpy as np
import time

MODEL_IDS = [
    "mms_tts_vie", "ntt123_viettts",
    "valtec_tts", "vieneu_v3nano", "vieneu_v3turbo","styletts2_lite_vi",
]
for m in MODEL_IDS:
    os.makedirs(f"{AUDIO_DIR}/{m}", exist_ok=True)

def run_batch(model_id, generate_fn):
    """Chạy generate_fn(text) -> (audio_np, sample_rate) cho toàn bộ PROMPTS, lưu wav + log thời gian."""
    timings = {}
    for pid, text in PROMPTS.items():
        out_path = f"{AUDIO_DIR}/{model_id}/{pid}.wav"
        if os.path.exists(out_path):
            continue  # đã sinh rồi, bỏ qua (resume nếu bị ngắt session)
        try:
            t0 = time.time()
            audio, sr = generate_fn(text)
            dt = time.time() - t0
            sf.write(out_path, audio, sr)
            timings[pid] = dt
        except Exception as e:
            print(f"[{model_id}] Lỗi ở câu '{pid}': {e}")
    print(f"[{model_id}] Hoàn tất. Thời gian trung bình/câu: {np.mean(list(timings.values())) if timings else 0:.2f}s")
    return timings

## 3. Model 1 — facebook/mms-tts-vie (VITS, không cloning)

In [ ]:
from transformers import VitsModel, AutoTokenizer
import torch

_mms_model = VitsModel.from_pretrained("facebook/mms-tts-vie")
_mms_tok = AutoTokenizer.from_pretrained("facebook/mms-tts-vie")

def generate_mms(text):
    inputs = _mms_tok(text, return_tensors="pt")
    with torch.no_grad():
        output = _mms_model(**inputs).waveform
    audio = output.squeeze().numpy()
    return audio, _mms_model.config.sampling_rate

run_batch("mms_tts_vie", generate_mms)

config.json:   0%|          | 0.00/1.64k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  145MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

[mms_tts_vie] Hoàn tất. Thời gian trung bình/câu: 0.00s


{}

## 5. Model 2 — NTT123/vietTTS (kiến trúc cũ, không cloning)

Kiểm tra lại github.com/NTT123/vietTTS nếu lệnh cài/API đã đổi — repo khá lâu chưa cập nhật.

In [ ]:
!git clone -q https://github.com/NTT123/vietTTS.git /content/vietTTS_ntt123
%cd /content/vietTTS_ntt123
!pip install -q -e .
%cd /content
import subprocess
import soundfile as sf

def generate_ntt123(text):
    out_path = "/content/_ntt123_tmp.wav"
    result = subprocess.run(
        ["python", "-m", "vietTTS.synthesizer", "--text", text, "--output", out_path],
        cwd="/content/vietTTS_ntt123",
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print("STDOUT:", result.stdout)
        print("STDERR:", result.stderr)
        raise RuntimeError(f"vietTTS CLI thất bại (mã {result.returncode})")
    audio, sr = sf.read(out_path)
    return audio, sr

run_batch("ntt123_viettts", generate_ntt123)

/content/vietTTS_ntt123
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.0/377.0 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 6.6 MB/s eta 0:00:00
/content
STDOUT: Normalized text input: wow sil đội tuyển vừa ghi bàn thắng đẹp quá sil

STDERR: Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/vietTTS_ntt123/vietTTS/synthesizer.py", line 36, in <module>
    mel = text2mel(text, args.lexicon_file, args.silence_duration)
  File "/content/vietTTS_ntt123/vietTTS/nat/text2mel.py", line 88, in text2mel
    tokens = text2tokens(text, lexicon_fn)
  File "/content/vietTTS_ntt123/vietTTS/nat/text2mel.py", line 39, in text2tokens
    lexicon = load_lexicon(lexicon_fn)
  File "/content/vietTTS_ntt123/vietTTS/nat/text2mel.py", line 17, in load_lexicon
    lines = open(fn, "r").

{}

## 7. Model 5 — VieNeu-TTS v3 Nano (48M, CPU-only cho máy yếu, có cloning)

In [ ]:
!pip install -q vieneu

from vieneu import Vieneu

_vieneu_nano = Vieneu(mode="v3nano")

def generate_vieneu_nano(text, ref_audio=REF_AUDIO_PATH):
    audio = _vieneu_nano.infer(text, ref_audio=ref_audio, denoise=True)
    return audio, 24000

run_batch("vieneu_v3nano", generate_vieneu_nano)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 32.9 MB/s eta 0:00:00


text_encoder.onnx: reconstructing file:   0%|          |  0.00B / 26.5MB            

text_encoder.onnx: downloading bytes:           |  0.00B            

duration_predictor.onnx: reconstructing file:   0%|          |  0.00B /  728kB            

duration_predictor.onnx: downloading bytes:           |  0.00B            

...

config.json:   0%|          | 0.00/2.93k [00:00<?, ?B/s]

constants.npz: reconstructing file:   0%|          |  0.00B / 53.4kB            

constants.npz: downloading bytes:           |  0.00B            

[vieneu_v3nano] Hoàn tất. Thời gian trung bình/câu: 0.00s


{}

## 8. Model 6 — VieNeu-TTS-v3-Turbo (0.1B, CPU tốt, có cloning)

In [ ]:
from vieneu import Vieneu

_vieneu_v3 = Vieneu(mode="v3turbo")  # mặc định CPU dùng ONNX tự động, không cần truyền backend=

def generate_vieneu_v3(text, ref_audio=REF_AUDIO_PATH):
    audio = _vieneu_v3.infer(text, ref_audio=ref_audio, denoise=True)
    return audio, 48000  # v3 Turbo 48kHz

run_batch("vieneu_v3turbo", generate_vieneu_v3)


config.json:   0%|          | 0.00/2.15k [00:00<?, ?B/s]

[transformers] You are using a model of type `vieneu_v3` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


tokenizer_config.json:   0%|          | 0.00/8.62k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/22.3k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.11k [00:00<?, ?B/s]

...

speaker_encoder.onnx: downloading bytes:           |  0.00B            

denoiser.onnx: reconstructing file:   0%|          |  0.00B / 42.7MB            

denoiser.onnx: downloading bytes:           |  0.00B            

[vieneu_v3turbo] Hoàn tất. Thời gian trung bình/câu: 0.00s


{}

In [ ]:
!pip install -q phonemizer soundfile
!apt-get -qq install -y espeak-ng
import subprocess, os, sys
import numpy as np

repo_dir = "/content/StyleTTS2-lite-vi"
if not os.path.exists(repo_dir):
    subprocess.run(["git", "clone", "https://huggingface.co/dangtr0408/StyleTTS2-lite-vi", repo_dir])
sys.path.append(repo_dir)

from inference import StyleTTS2

_device = "cuda" if torch.cuda.is_available() else "cpu"
_style_config = os.path.join(repo_dir, "Models", "config.yaml")
_style_ckpt = os.path.join(repo_dir, "Models", "inference", "model.pth")
_style_model = StyleTTS2(_style_config, _style_ckpt).eval().to(_device)

def generate_styletts2(text, ref_audio=REF_AUDIO_PATH):
    speakers = {"id_1": {"path": ref_audio, "lang": "vi", "speed": 1.0}}
    styles = _style_model.get_styles(speakers, denoise=0.2, avg_style=True)
    audio = _style_model.generate(text, styles, True, 18, "[id_1]")
    audio = audio / np.abs(audio).max()  # chuẩn hoá biên độ
    return audio, 24000

run_batch("styletts2_lite_vi", generate_styletts2)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.5/108.5 kB 2.7 MB/s eta 0:00:00
Selecting previously unselected package libpcaudio0:amd64.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../libpcaudio0_1.2-2build3_amd64.deb ...
Unpacking libpcaudio0:amd64 (1.2-2build3) ...
Selecting previously unselected package libsonic0:amd64.
Preparing to unpack .../libsonic0_0.2.0-13build1_amd64.deb ...
Unpacking libsonic0:amd64 (0.2.0-13build1) ...
Selecting previously unselected package espeak-ng-data:amd64.
Preparing to unpack .../espeak-ng-data_1.51+dfsg-12build1_amd64.deb ...
Unpacking espeak-ng-data:amd64 (1.51+dfsg-12build1) ...
Selecting previously unselected package libespeak-ng1:amd64.
Preparing to unpack .../libespeak-ng1_1.51+dfsg-12build1_amd64.deb ...
Unpacking libespeak-ng1:amd64 (1.51+dfsg-12build1) ...
Selecting previously unselected package espeak-ng.
Prepa

ModuleNotFoundError: No module named 'munch'

## 10. Đánh giá WER/CER — dùng PhoWhisper

In [ ]:
from transformers import pipeline
import jiwer

asr = pipeline("automatic-speech-recognition", model="vinai/PhoWhisper-small", device=0 if torch.cuda.is_available() else -1)

def normalize_vi(s):
    return s.lower().strip()

wer_results = []
for model_id in MODEL_IDS:
    for pid, ref_text in PROMPTS.items():
        wav_path = f"{AUDIO_DIR}/{model_id}/{pid}.wav"
        if not os.path.exists(wav_path):
            continue
        hyp_text = asr(wav_path)["text"]
        wer = jiwer.wer(normalize_vi(ref_text), normalize_vi(hyp_text))
        wer_results.append({"model": model_id, "prompt_id": pid, "ref": ref_text, "hyp": hyp_text, "wer": wer})

import pandas as pd
df_wer = pd.DataFrame(wer_results)
df_wer.to_csv(f"{RESULT_DIR}/wer_detail.csv", index=False)

wer_summary = df_wer.groupby("model")["wer"].mean().sort_values()
print("=== WER trung bình theo model (thấp hơn = tốt hơn) ===")
print(wer_summary)
wer_summary.to_csv(f"{RESULT_DIR}/wer_summary.csv")

## 11. Đánh giá UTMOS (độ tự nhiên tự động)

In [ ]:
!pip install -q git+https://github.com/tarepan/SpeechMOS.git

import torch, librosa
predictor = torch.hub.load("tarepan/SpeechMOS:v1.2.0", "utmos22_strong", trust_repo=True)

utmos_results = []
for model_id in MODEL_IDS:
    for pid in PROMPTS:
        wav_path = f"{AUDIO_DIR}/{model_id}/{pid}.wav"
        if not os.path.exists(wav_path):
            continue
        wave, sr = librosa.load(wav_path, sr=16000)
        score = predictor(torch.from_numpy(wave).unsqueeze(0), sr).item()
        utmos_results.append({"model": model_id, "prompt_id": pid, "utmos": score})

df_utmos = pd.DataFrame(utmos_results)
df_utmos.to_csv(f"{RESULT_DIR}/utmos_detail.csv", index=False)

utmos_summary = df_utmos.groupby("model")["utmos"].mean().sort_values(ascending=False)
print("=== UTMOS trung bình theo model (cao hơn = tự nhiên hơn, chỉ mang tính tham khảo) ===")
print(utmos_summary)
utmos_summary.to_csv(f"{RESULT_DIR}/utmos_summary.csv")

## 12. Đánh giá SIM-o (giống giọng mẫu — chỉ cho model có cloning)

Cần file `REF_AUDIO_PATH` đã upload ở bước 1 (mono, 3-10s, .wav). Dùng ECAPA-TDNN (speechbrain) để trích embedding + tính cosine similarity.

In [ ]:
from speechbrain.inference.speaker import EncoderClassifier
import torch.nn.functional as F

spk_model = EncoderClassifier.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb")

CLONING_MODELS = ["valtec_tts", "vieneu_v3nano", "vieneu_v3turbo"]  # 3 model có ref_audio

def get_embedding(wav_path):
    signal = spk_model.load_audio(wav_path)
    emb = spk_model.encode_batch(signal.unsqueeze(0))
    return emb.squeeze()

ref_emb = get_embedding(REF_AUDIO_PATH)

simo_results = []
for model_id in CLONING_MODELS:
    for pid in PROMPTS:
        wav_path = f"{AUDIO_DIR}/{model_id}/{pid}.wav"
        if not os.path.exists(wav_path):
            continue
        emb = get_embedding(wav_path)
        sim = F.cosine_similarity(ref_emb, emb, dim=0).item()
        simo_results.append({"model": model_id, "prompt_id": pid, "sim_o": sim})

df_simo = pd.DataFrame(simo_results)
df_simo.to_csv(f"{RESULT_DIR}/simo_detail.csv", index=False)

simo_summary = df_simo.groupby("model")["sim_o"].mean().sort_values(ascending=False)
print("=== SIM-o trung bình theo model (cao hơn = giống giọng mẫu hơn) ===")
print(simo_summary)
simo_summary.to_csv(f"{RESULT_DIR}/simo_summary.csv")

## 13. Tổng hợp — trình bày riêng từng bảng (không gộp điểm có trọng số)

Theo đúng cách paper OmniVoice làm: mỗi metric là 1 bảng riêng, không cộng dồn có trọng số tự đặt.

In [ ]:
print("=" * 50)
print("BẢNG 1: WER trung bình")
print("=" * 50)
print(wer_summary)

print()
print("=" * 50)
print("BẢNG 2: UTMOS trung bình")
print("=" * 50)
print(utmos_summary)

print()
print("=" * 50)
print("BẢNG 3: SIM-o trung bình")
print("=" * 50)
print(simo_summary)

print()
print("→ RTF và CMOS/SMOS: đo/thu thập riêng")
print(f"Toàn bộ chi tiết đã lưu tại: {RESULT_DIR}")

In [ ]:
from vieneu import Vieneu
Vieneu(mode="v3turbo")